In [1]:
# =========================================================
# Scenario Answer: AI-Powered Report Assistant (HITL-Based)
# =========================================================
# Imagine designing an AI-powered assistant that drafts structured reports,
# pauses for human review, and then either publishes or rejects them.
#
# To ensure accountability and human oversight, the system can be structured
# using a state-driven workflow graph with clearly defined nodes and transitions.
#
# -------------------------
# 1. State Design
# -------------------------
# The system maintains a shared state that acts like a notebook:
#
# - task         → the topic or purpose of the report
# - draft        → the AI-generated structured report
# - approved     → a boolean flag representing human decision
# - feedback     → reviewer comments for improving the draft
# - final_status → final result (PUBLISHED or REJECTED)
#
# This state ensures that all intermediate and final decisions are tracked,
# making the system auditable and transparent.
#
# -------------------------
# 2. Workflow Graph
# -------------------------
# The workflow is modeled as a graph of states with the following nodes:
#
# (1) Draft Node:
#     - The AI generates a structured report based on the given task.
#     - The draft includes sections like title, introduction, key points, and conclusion.
#     - The output is stored in the "draft" field.
#
# (2) Review Node (Human-in-the-Loop):
#     - The workflow pauses before this node using an interrupt mechanism.
#     - A human reviewer inspects the generated draft.
#     - The reviewer can:
#         a) Approve the report
#         b) Reject the report
#         c) Provide feedback for revision
#     - The state is updated with "approved" and "feedback".
#
# (3) Conditional Flow:
#     - If approved = True → move to Publish Node
#     - If approved = False:
#           → If feedback exists → return to Draft Node for revision
#           → Else → directly reject
#
# (4) Publish Node:
#     - If approved → report is published (final_status = PUBLISHED)
#     - If rejected → report is not published (final_status = REJECTED)
#
# (5) End Node:
#     - The workflow completes with a clear final outcome.
#
# -------------------------
# 3. Graph Flow Representation
# -------------------------
#
# task input
#    ↓
# draft (AI generation)
#    ↓
# ⏸ review (human approval checkpoint)
#    ↓
# decision:
#    ├── approved → publish → END
#    └── not approved
#           ├── feedback → redraft → review (loop)
#           └── no feedback → reject → END
#
# -------------------------
# 4. Ensuring Accountability & Oversight
# -------------------------
# This design ensures accountability in multiple ways:
#
# - Human Approval Gate:
#   AI cannot publish reports independently; every output requires human validation.
#
# - Interrupt Mechanism:
#   The workflow explicitly pauses before the review node, enforcing oversight.
#
# - Feedback Loop:
#   Reviewers can improve the report instead of just accepting/rejecting it.
#
# - State Tracking:
#   All decisions (approval, rejection, feedback) are stored in the state,
#   enabling auditability and traceability.
#
# - Deterministic Outcome:
#   The system guarantees a clear final state: either PUBLISHED or REJECTED.
#
# -------------------------
# 5. Conclusion
# -------------------------
# By combining AI-driven drafting with human-in-the-loop validation,
# this architecture balances automation with control, ensuring that
# reports are accurate, reliable, and approved before publication.
#
# This makes the system suitable for real-world enterprise use cases
# such as business reporting, compliance documentation, and research analysis.
# =========================================================

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import requests
from google.colab import userdata

# =========================
# State
# =========================
class ReportState(TypedDict):
    task: str
    draft: str
    approved: bool
    feedback: str
    final_status: str


# =========================
# Groq API Call
# =========================
def groq_call(prompt):
    api_key = userdata.get("groq_api_key")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}]
        }
    )

    return response.json()["choices"][0]["message"]["content"]


# =========================
# Node 1: Draft Report
# =========================
def draft_report(state: ReportState):

    # If feedback exists → revise report
    if state.get("feedback"):
        prompt = f"""
Revise the following report using reviewer feedback.

Topic: {state['task']}

Report:
{state['draft']}

Feedback:
{state['feedback']}

Make it structured and professional.
"""
    else:
        prompt = f"""
Write a structured professional report.

Topic: {state['task']}

Include:
- Title
- Introduction
- Key Points
- Conclusion
"""

    draft = groq_call(prompt)

    print("\n📄 Draft Report:\n", draft)

    return {"draft": draft}


# =========================
# Node 2: Human Review
# =========================
def review(state: ReportState):
    print("\n⏸ Waiting for human review...\n")
    print(state["draft"])
    return {}


# =========================
# Node 3: Publish / Reject
# =========================
def publish(state: ReportState):
    if state["approved"]:
        print("\n✅ Report Published")
        return {"final_status": "PUBLISHED"}
    else:
        print("\n❌ Report Rejected")
        return {"final_status": "REJECTED"}


# =========================
# Graph
# =========================
g = StateGraph(ReportState)

g.add_node("draft", draft_report)
g.add_node("review", review)
g.add_node("publish", publish)

g.set_entry_point("draft")
g.add_edge("draft", "review")
g.add_edge("review", "publish")
g.add_edge("publish", END)

# =========================
# HITL Checkpoint
# =========================
app = g.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["review"]
)

# =========================
# Run
# =========================
thread = {"configurable": {"thread_id": "report-1"}}

# Step 1: Generate draft
app.invoke({
    "task": input("Enter report topic: "),
    "draft": "",
    "approved": False,
    "feedback": "",
    "final_status": ""
}, thread)

# Step 2: Human review input
decision = input("Approve report? (yes/no): ")
feedback = input("Feedback (optional): ")

# Optional re-draft
if decision == "no" and feedback:
    app.invoke({
        "approved": False,
        "feedback": feedback
    }, thread)

    print("\n🔁 Revised report generated.\n")

    decision = input("Approve revised report? (yes/no): ")

# Final step
result = app.invoke({
    "approved": True if decision == "yes" else False
}, thread)

print("\nFinal Status:", result["final_status"])

Enter report topic: mindfulness

📄 Draft Report:
 **Title:** The Benefits of Mindfulness: Enhancing Mental and Physical Well-being in the Modern Era

**Introduction:**

In today's fast-paced and increasingly stressful world, the importance of maintaining mental and physical well-being cannot be overstated. With the constant demands of work, social media, and personal life, it is easy to get caught up in a never-ending cycle of anxiety, depression, and burnout. However, there is a simple yet powerful tool that can help individuals break free from this cycle: mindfulness. This report aims to explore the benefits of mindfulness, its key principles, and how it can be effectively incorporated into daily life to improve overall well-being.

**Key Points:**

1. **Definition and Principles of Mindfulness:** Mindfulness is the practice of being fully present and engaged in the current moment, while cultivating a non-judgmental awareness of one's thoughts, feelings, and physical sensations. The 